<a href="https://colab.research.google.com/github/mrityunjaykumar911/TorchCode/blob/master/templates/02_softmax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/02_softmax.ipynb)

# 🟢 Easy: Implement Softmax

Implement the **Softmax** function from scratch.

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

### Signature
```python
def my_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    ...
```

### Rules
- Do **NOT** use `torch.softmax`, `F.softmax`, or `torch.nn.Softmax`
- Must be **numerically stable** (hint: subtract `max` before `exp`)

### Example
```
Input:  tensor([1., 2., 3.])
Output: tensor([0.0900, 0.2447, 0.6652])  # sums to 1.0
```

Softmax is the standard bridge between unconstrained real-valued scores and a probability distribution. Networks naturally produce arbitrary numbers (logits), but many decisions need probabilities: positive values that sum to 1. Softmax performs that conversion in a way that is smooth, differentiable, and has good gradients.


1. It turns scores into a distribution while keeping their order.
Exponentiation makes every value positive and preserves ranking. Normalizing makes the values sum to 1. It also amplifies gaps: a logit that is 2 higher gets about 7.4× the probability. So the output behaves like a “soft argmax,” hence the name.

2. Its gradient with cross-entropy is very clean.
For softmax followed by negative log-likelihood, the gradient with respect to the logits is simply p − y (predicted distribution minus one-hot target). That makes classification training stable and cheap. It also explains why frameworks fuse softmax and the loss into one operation.

3. It is the maximum-entropy distribution for given scores.
Softmax is the Boltzmann/Gibbs distribution from statistical physics. Among all distributions with a fixed expected score, it is the one with maximum entropy. In other words, it is the least-assuming way to turn preferences into probabilities. This is why it comes up in principled settings, not just as a convenient trick.

4. Temperature gives you an exploration knob.
softmax(x / T): as T → 0 it approaches a hard argmax (greedy), and as T → ∞ it approaches uniform. This single parameter controls LLM sampling and Boltzmann exploration in RL.

5. It shows up in nearly every modern architecture.

Attention: softmax(QKᵀ/√d) turns similarity scores into mixing weights over tokens. Each transformer layer runs it many times.
LLM output: each next-token prediction is a softmax over the vocabulary.
RL policies: a softmax policy π(a|s) ∝ exp(θ·φ(s,a)) gives the clean ∇log π used in REINFORCE and PPO (Sutton & Barto, ch. 13). Soft Q-learning and SAC use the same exponential form, linked to entropy regularization.
Contrastive learning (CLIP, InfoNCE): softmax over similarities picks out the positive pair from the negatives.

The core idea: softmax is how a model expresses “I prefer A over B by this much” in a form you can sample from, take gradients through, and compare against targets.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

In [16]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
   _max = torch.max(x, dim=dim, keepdim=True).values
   x = x - _max
   numerator = torch.exp(x)
   denominator = torch.sum(numerator, dim=dim, keepdim=True)
   return numerator / denominator

In [17]:
# 🧪 Debug
x = torch.tensor([1.0, 2.0, 3.0])
print("Output:", my_softmax(x, dim=-1))
print("Sum:   ", my_softmax(x, dim=-1).sum())  # should be ~1.0
print("Ref:   ", torch.softmax(x, dim=-1))

Output: tensor([0.0900, 0.2447, 0.6652])
Sum:    tensor(1.)
Ref:    tensor([0.0900, 0.2447, 0.6652])


In [18]:
# ✅ SUBMIT
from torch_judge import check
check("softmax")


🧪 Testing: Implement Softmax (Easy)
──────────────────────────────────────────────────
  ✅ [1/3] Basic 1-D (2.6ms)
  ✅ [2/3] 2-D along dim=-1 (2.4ms)
  ✅ [3/3] Numerical stability (1.0ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (6.0ms total)
  Progress saved. Run status() to see your dashboard.

